This notebook is implemented to split the __Gowalla__ and __Tmall__ datasets into training/validation/test sets, the datasets are given by the authors of the [LightGCL](https://openreview.net/forum?id=FKXVK9dyMM) paper on their [Github repository](https://github.com/HKUDS/LightGCL?tab=readme-ov-file). The reason we use these datasets is to ensure the fairness between __DiffRec__ with 3 selected alternative methods _LightGCN_, _SimGCL_ and _LightGCL_ (which are already tested on these 2 datasets, their results are shown in the __Table 1__ in the LightGCL paper).

Before running the code, please ensure all necessary packages are installed. The list of packages can be found inside the `requirements.txt`. Besides that, please download the datasets from LightGCL repository, then adjust the path below to the directory containing `trnMat.pkl` file:

In [1]:
gowalla_path = "./gowalla/"
tmall_path = "./tmall/"

# Import packages

In [2]:
import pickle
import os
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix

# Fix the seed for reproducibility

In [3]:
seed = 24
np.random.seed(seed)

# Load data from files

In [4]:
# Helper function to read pickle files
def read_pkl(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    
    return data

In [5]:
gowalla_mat = read_pkl(gowalla_path + "trnMat.pkl")
gowalla_test = read_pkl(gowalla_path + "tstMat.pkl")
tmall_mat = read_pkl(tmall_path + "trnMat.pkl")
tmall_test = read_pkl(tmall_path + "tstMat.pkl")

C:\Users\Sn\AppData\Local\Temp\ipykernel_38860\374016073.py:4: DeprecationWarning: Please import `coo_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.coo` namespace is deprecated and will be removed in SciPy 2.0.0.
  data = pickle.load(file)
C:\Users\Sn\AppData\Local\Temp\ipykernel_38860\374016073.py:4: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath._reconstruct.
  data = pickle.load(file)
C:\Users\Sn\AppData\Local\Temp\ipykernel_38860\374016073.py:4: VisibleDeprecationWarning: dtype(

# Overlapping check

In [6]:
print("Gowalla overlapping: ", gowalla_mat.multiply(gowalla_test).nnz)
print("Tmall overlapping: ", tmall_mat.multiply(tmall_test).nnz)

Gowalla overlapping:  0
Tmall overlapping:  0


There is no co-existed elements in both TrnMat and TstMat for all datasets.

# Create Validation set

Because DiffRec requires a validation sest but the authors of LightGCL didn't release the validation set explicitly, we will split the training set of each dataset into training and validation sets with a ratio of 8:2, and convert to Compressed Sparse Row matrix format.

In [7]:
def train_val_split(data_mat: csr_matrix, val_ratio=0.2):
    """
    Perform train-validation splits on the given CSR matrix
    """

    # Convert to DataFrame
    df = pd.DataFrame({
        "user": data_mat.row,
        "item": data_mat.col,
        "value": data_mat.data
    })

    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Rank interactions within each user
    df["rank"] = df.groupby("user").cumcount()

    # Count interactions per user
    df["count"] = df.groupby("user")["item"].transform("count")

    # Number of validation items per user (at least 1 if possible)
    val_cut = (df["count"] * val_ratio).astype(int)
    val_cut = val_cut.clip(lower=1)

    # Keep users with too few interactions in train
    df["is_val"] = (df["count"] > 1) & (df["rank"] < val_cut)

    # Split
    val_df = df[df["is_val"]]
    train_df = df[~df["is_val"]]

    # Convert back to CSR
    shape = data_mat.shape

    train_csr = coo_matrix(
        (train_df["value"], (train_df["user"], train_df["item"])),
        shape=shape
    ).tocsr()

    val_csr = coo_matrix(
        (val_df["value"], (val_df["user"], val_df["item"])),
        shape=shape
    ).tocsr()

    return train_csr, val_csr

In [8]:
gowalla_train, gowalla_val = train_val_split(gowalla_mat, val_ratio=0.2)

In [9]:
tmall_train, tmall_val = train_val_split(tmall_mat, val_ratio=0.2)

## Statistics check 

In [10]:
print("Gowalla")
print("Type  - Users - Items - Interactions")
print(f"Train\t{gowalla_train.shape[0]}\t{gowalla_train.shape[1]}\t{gowalla_train.data.shape[0]}")
print(f"Valid\t{gowalla_val.shape[0]}\t{gowalla_val.shape[1]}\t{gowalla_val.data.shape[0]}")
print(f"Test \t{gowalla_test.shape[0]}\t{gowalla_val.shape[1]}\t{gowalla_test.data.shape[0]}")

Gowalla
Type  - Users - Items - Interactions
Train	50821	57440	955363
Valid	50821	57440	217062
Test 	50821	57440	130270


In [11]:
print("Tmall")
print("Type  - Users - Items - Interactions")
print(f"Train\t{tmall_train.shape[0]}\t{tmall_train.shape[1]}\t{tmall_train.data.shape[0]}")
print(f"Valid\t{tmall_val.shape[0]}\t{tmall_val.shape[1]}\t{tmall_val.data.shape[0]}")
print(f"Test \t{tmall_test.shape[0]}\t{tmall_test.shape[1]}\t{tmall_test.data.shape[0]}")

Tmall
Type  - Users - Items - Interactions
Train	47939	41390	1905226
Valid	47939	41390	452224
Test 	47939	41390	261939


# Save to files

In [12]:
def save_sparse(sparse_mat, filename, dir_name=""):
    # Save to file for DiffRec input
    # save_npz(f"{path}/{filename}.npz", sparse_mat)

    rows, cols = sparse_mat.nonzero()

    data_to_save = np.stack([rows, cols], axis=1)

    np.save(f"{dir_name}/{filename}", data_to_save)

In [13]:
save_sparse(gowalla_train, 'train_list', gowalla_path)
save_sparse(gowalla_val, 'valid_list', gowalla_path)
save_sparse(gowalla_test, 'test_list', gowalla_path)

save_sparse(tmall_train, 'train_list', tmall_path)
save_sparse(tmall_val, 'valid_list', tmall_path)
save_sparse(tmall_test, 'test_list', tmall_path)